# Notebook 11: first model call (thinking ON vs OFF)
Lesson: [11 First model call](../lessons/11-first-model-call.md).

**Needs a GPU. Run it on Kaggle, not Colab** (see "Why Kaggle" below).

**First, once per account:** Kaggle → your profile picture → **Settings** → **Phone verification**. Until you do this, "GPU T4 x2" is grey and cannot be chosen, and Internet stays off.

**On Kaggle:** New Notebook → File → Import Notebook → upload this file. Then, in the right panel under **Settings**:
- **Accelerator → GPU T4 x2** (we use one GPU; the second one is the backup plan).
- **Internet → On**. Kaggle asks you to check your phone number once. Without internet, the install and the model download fail.
- Then run the cells from top to bottom, **once each**. Expect 20–40 minutes, mostly installing and downloading.

**What this notebook does:** asks Gemma-4-E4B one easy code question, twice with thinking ON and twice with thinking OFF, counts the thinking tokens, and saves every raw answer.

**What it does NOT do:** it never runs the code that the model writes. Running model-written code happens later, in a sandbox (lesson 14).

**Why Kaggle (2026-09-19):** on a free Colab T4, loading the model crashed with "out of memory" (twice). The cause and the fix are explained in section 5 and in `results/2026-09-19-notebook11-out-of-memory.md`. The fix needs about 16 GB of normal computer memory (CPU memory). Free Colab has about 12.7 GB; Kaggle has about 29 GB (not checked yet).

⚠️ This notebook has **not run to the end yet**. If a cell fails, that is a finding: write down the error.

## 1. Which GPU did we get?
**Problem:** the free GPU changes. **Why:** memory numbers only make sense with the GPU name.
**In:** nothing. **Out:** GPU name and memory. **Why this way:** `nvidia-smi` is the standard check.

In [ ]:
!nvidia-smi

## 2. Install the libraries
**Problem:** Colab doesn't have Unsloth. **Why:** Unsloth loads Gemma 4 in 4-bit on a small GPU.
**In:** nothing. **Out:** installed libraries. **Why this way:** copied from Unsloth's official Gemma 4 T4 notebook, which fixes the versions that work together.

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
import torch; torch._dynamo.config.recompile_limit = 64

## 3. All settings in one place
**Problem:** every run must be repeatable. **Why:** fixed seeds and one place for limits (CLAUDE.md §4).
**In:** nothing. **Out:** settings used by every cell below. **Why this way:** one cell to change, nothing hidden.

The sampling settings come from Gemma's model card ("Best Practices"). The same settings are used for thinking ON and OFF, so the comparison is fair.

In [ ]:
MODEL_ID       = "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit"   # the 4-bit version, 10.95 GB
SEED           = 3407          # fixed, so the run can be repeated
N_TRIES        = 2             # how many answers per way of answering
MAX_NEW_TOKENS = 4096          # the same limit for ON and OFF
GEN_KWARGS     = dict(do_sample=True, temperature=1.0, top_p=0.95, top_k=64)   # Gemma's recommended settings

# How to fit the model (see section 5). Try "table_on_cpu" first.
# If loading or asking fails: Run → Restart, change this to "two_gpus", and run from the top again.
LOAD_MODE      = "table_on_cpu"   # "table_on_cpu" or "two_gpus"

QUESTION = (
    "Write a Python function is_palindrome(s) that returns True if the string s "
    "reads the same forwards and backwards, ignoring upper and lower case. "
    "Answer with the function only."
)
PROBLEM_ID = "lesson11_is_palindrome"   # our own question, NOT a test-set problem

## 4. Where results are saved
**Problem:** a free session can die at any second. **Why:** we must never lose a finished answer.
**In:** nothing. **Out:** a file path on Google Drive (Colab) or in the working folder. **Why this way:** append-only file on storage that outlives the session (PLAN.md §11).

In [ ]:
import os, json, time

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/stop-overthinking/results"
except Exception as e:
    print("No Google Drive (that is fine on Kaggle/local):", e)
    OUT_DIR = "results"

os.makedirs(OUT_DIR, exist_ok=True)
OUT_FILE = os.path.join(OUT_DIR, "11_first_model_call.jsonl")
print("saving to:", OUT_FILE)

## 5. Load the model
**Problem:** the model must fit in a T4 GPU (about 14.56 GB usable). **Why:** lesson 07.
**In:** the model id and `LOAD_MODE`. **Out:** the model and its tokenizer, plus the GPU and CPU memory used. **Why this way:** see below.

**What went wrong on Colab (2026-09-19, checked).** Gemma-4-E4B has a big *per-layer word table*: 262,144 words × 42 layers × 256 numbers × 2 bytes = **5.25 GB**. It is not squeezed to 4-bit. The T4 cannot compute in bf16, so Unsloth converts this table to another 16-bit format (float16) **on the GPU**. For a moment it needs two copies: 10.22 GB + 5.25 GB > 14.56 GB → out of memory. (Unsloth's older "9.891 GB after loading" is not true for Unsloth 2026.9.7.)

**The fix (`table_on_cpu`).** Keep that one table in normal computer memory (CPU memory), and everything else on the GPU. Each word only needs one row of the table, so this is cheap. Two small "hooks" move the data: word ids go to the CPU table, and its rows come back to the GPU.

**Backup (`two_gpus`).** Split the model over Kaggle's two T4s. Then no single GPU is full during the conversion.

**Not checked yet:** whether each mode works end to end. This cell and section 8 test it.

The download is about 11 GB, so this cell can take 10–20 minutes the first time.

In [ ]:
from unsloth import FastModel
from huggingface_hub import snapshot_download
import torch, psutil, os, json

TABLE = "model.language_model.embed_tokens_per_layer"   # the 5.25 GB per-layer word table (name checked in the model files)

def model_folder_with_cpu_flag(model_id):
    """transformers refuses a CPU part in a 4-bit model unless llm_int8_enable_fp32_cpu_offload is on.
    For an already-4-bit model it reads that flag ONLY from the model's own config.json (checked in its
    source code, 2026-09-20), so a flag passed in code is ignored. We make a folder of shortcuts to the
    downloaded files, with one changed config.json. No second download, no extra disk space."""
    src = snapshot_download(model_id)                        # downloads ~11 GB the first time; later runs in the same session find it on disk
    dst = "/tmp/" + model_id.split("/")[-1]                  # keep the name: Unsloth reads "bnb-4bit" from it
    os.makedirs(dst, exist_ok=True)
    for f in os.listdir(src):
        if f != "config.json" and not os.path.exists(os.path.join(dst, f)):
            os.symlink(os.path.join(src, f), os.path.join(dst, f))
    cfg = json.load(open(os.path.join(src, "config.json")))
    cfg["quantization_config"]["llm_int8_enable_fp32_cpu_offload"] = True
    json.dump(cfg, open(os.path.join(dst, "config.json"), "w"), indent=2)
    return dst

if LOAD_MODE == "table_on_cpu":
    model_source = model_folder_with_cpu_flag(MODEL_ID)
    device_map   = {"": 0, TABLE: "cpu"}
elif LOAD_MODE == "two_gpus":
    model_source, device_map = MODEL_ID, "balanced"
else:
    raise ValueError("LOAD_MODE must be 'table_on_cpu' or 'two_gpus'")

model, tokenizer = FastModel.from_pretrained(
    model_name      = model_source,
    dtype           = None,      # None = decide automatically (the T4 has no bf16)
    max_seq_length  = 8192,
    load_in_4bit    = True,
    full_finetuning = False,
    device_map      = device_map,
)

if LOAD_MODE == "table_on_cpu":
    from accelerate.hooks import remove_hook_from_module
    table = dict(model.named_modules())[TABLE]
    remove_hook_from_module(table)   # our own two hooks below do the moving instead
    # the model adds this table's rows to numbers on the GPU, so: ids go to the CPU, rows come back to the GPU
    table.register_forward_pre_hook(lambda m, args: (args[0].to(m.weight.device),) + tuple(args[1:]))
    table.register_forward_hook(lambda m, args, out: out.to("cuda:0"))
    print("per-layer table is on:", table.weight.device, table.weight.dtype)

text_tok = getattr(tokenizer, "tokenizer", tokenizer)   # the plain text tokenizer inside
for i in range(torch.cuda.device_count()):
    print(f"GPU {i} memory used after loading (GB):", round(torch.cuda.max_memory_reserved(i) / 1024**3, 3))
print("CPU memory used by this notebook (GB):", round(psutil.Process().memory_info().rss / 1024**3, 3))

## 6. Does the thinking switch really change the prompt?
**Problem:** we must be sure `enable_thinking` does something. **Why:** our whole thesis compares ON with OFF.
**In:** our question. **Out:** the two prompts, and a check that fails loudly if they are the same. **Why this way:** check the tool before trusting it.

Gemma's chat template puts the special token `<|think|>` in the prompt when thinking is ON (model card, checked 2026-09-17). Note: thinking is **OFF by default**, so we always set it on purpose.

In [ ]:
MESSAGES = [{"role": "user", "content": [{"type": "text", "text": QUESTION}]}]

def prompt_text(thinking):
    return tokenizer.apply_chat_template(
        MESSAGES, add_generation_prompt=True, tokenize=False, enable_thinking=thinking
    )

on_text, off_text = prompt_text(True), prompt_text(False)
print("--- thinking ON prompt ---\n", on_text)
print("--- thinking OFF prompt ---\n", off_text)

assert "<|think|>" in on_text, "Thinking ON did not add the <|think|> token. Stop and check the template."
assert "<|think|>" not in off_text, "Thinking OFF still has the <|think|> token. Stop and check the template."
print("\n✅ the switch changes the prompt")

## 7. Ask the model once, and count the tokens
**Problem:** we need thinking tokens, answer text and time per answer. **Why:** these are the numbers our thesis reports.
**In:** thinking ON/OFF and a seed. **Out:** one result record (raw text kept word for word). **Why this way:** count tokens from the model's own output ids (exact), and keep the raw text so re-checking never needs the GPU again.

In [ ]:
from transformers import set_seed

THINK_START, THINK_END = "<|channel>", "<channel|>"

def token_id_of(piece):
    i = text_tok.convert_tokens_to_ids(piece)
    return i if text_tok.convert_ids_to_tokens(i) == piece else None

START_ID, END_ID = token_id_of(THINK_START), token_id_of(THINK_END)

def ask(thinking, sample_index):
    seed = SEED + sample_index                 # same seeds for ON and OFF
    set_seed(seed)
    inputs = tokenizer.apply_chat_template(
        MESSAGES, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt", enable_thinking=thinking,
    ).to("cuda")

    # make sure the tokenized prompt really matches the switch
    assert ("<|think|>" in text_tok.decode(inputs["input_ids"][0])) == thinking

    prompt_len = inputs["input_ids"].shape[-1]
    t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, **GEN_KWARGS)
    seconds = time.time() - t0

    new_ids = out[0][prompt_len:].tolist()
    raw = text_tok.decode(new_ids, skip_special_tokens=False)

    if THINK_START in raw and THINK_END in raw:
        thinking_text = raw.split(THINK_START, 1)[1].split(THINK_END, 1)[0]
        answer_text = raw.split(THINK_END, 1)[1]
    else:
        thinking_text, answer_text = "", raw

    if START_ID in new_ids and END_ID in new_ids:      # exact count, from the ids
        thinking_tokens = new_ids.index(END_ID) - new_ids.index(START_ID) - 1
    else:                                              # fallback: count the text again
        thinking_tokens = len(text_tok(thinking_text, add_special_tokens=False)["input_ids"])

    return {
        "problem_id": PROBLEM_ID,
        "policy": "thinking_on" if thinking else "thinking_off",
        "sample_index": sample_index,
        "seed": seed,
        "model_id": MODEL_ID,
        "max_new_tokens": MAX_NEW_TOKENS,
        "gen_kwargs": GEN_KWARGS,
        "total_new_tokens": len(new_ids),
        "thinking_tokens": thinking_tokens,
        "hit_limit": len(new_ids) >= MAX_NEW_TOKENS,
        "seconds": round(seconds, 1),
        "tokens_per_second": round(len(new_ids) / seconds, 1),
        "raw_output": raw,              # kept word for word (CLAUDE.md §4)
        "answer_text": answer_text,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }

## 8. Run all four answers (and skip what is already done)
**Problem:** the session may die in the middle. **Why:** we then continue instead of starting again.
**In:** the settings. **Out:** 4 records appended to the results file. **Why this way:** the "done list + append + flush" pattern from PLAN.md §11, in its smallest form.

In [ ]:
done = set()
if os.path.exists(OUT_FILE):
    with open(OUT_FILE) as f:
        for line in f:
            r = json.loads(line)
            done.add((r["problem_id"], r["policy"], r["sample_index"]))
print("already done:", len(done))

results = []
for thinking in [True, False]:
    for i in range(N_TRIES):
        key = (PROBLEM_ID, "thinking_on" if thinking else "thinking_off", i)
        if key in done:
            print("skip (already done):", key); continue
        r = ask(thinking, i)
        results.append(r)
        with open(OUT_FILE, "a") as f:                 # append only
            f.write(json.dumps(r) + "\n")
            f.flush(); os.fsync(f.fileno())            # really save it to disk
        print(key, "→", r["thinking_tokens"], "thinking tokens,", r["seconds"], "seconds")

## 9. The result table
**Problem:** we want one small table to read. **Why:** this is the first evidence in the whole thesis.
**In:** the records we just made. **Out:** a printed table. **Why this way:** plain printing, no extra library.

In [ ]:
rows = results if results else [json.loads(l) for l in open(OUT_FILE)]
print(f"{'policy':<14}{'try':<5}{'thinking':>10}{'total':>8}{'seconds':>9}{'tok/s':>7}{'hit limit':>11}")
for r in rows:
    print(f"{r['policy']:<14}{r['sample_index']:<5}{r['thinking_tokens']:>10}{r['total_new_tokens']:>8}"
          f"{r['seconds']:>9}{r['tokens_per_second']:>7}{str(r['hit_limit']):>11}")

on = [r for r in rows if r["policy"] == "thinking_on"]
off = [r for r in rows if r["policy"] == "thinking_off"]
if on and off:
    avg = lambda xs, k: sum(x[k] for x in xs) / len(xs)
    print("\naverage thinking tokens  ON :", round(avg(on, "thinking_tokens"), 1))
    print("average thinking tokens  OFF:", round(avg(off, "thinking_tokens"), 1))
    print("average total tokens     ON :", round(avg(on, "total_new_tokens"), 1))
    print("average total tokens     OFF:", round(avg(off, "total_new_tokens"), 1))

## 10. Read one thinking part with your own eyes
**Problem:** numbers alone don't show overthinking. **Why:** lesson 05 asks you to recognise it.
**In:** the first thinking-ON answer. **Out:** its thinking text and its final answer. **Why this way:** reading one real example teaches more than a table.

In [ ]:
r = on[0] if on else rows[0]
print("=== THINKING (", r["thinking_tokens"], "tokens ) ===")
print(r["raw_output"].split(THINK_END)[0][:3000])
print("\n=== FINAL ANSWER ===")
print(r["answer_text"][:1500])

## 11. Memory and speed (write these numbers down)
**Problem:** PLAN.md §10 says our speed number is not checked. **Why:** this is our first real measurement.
**In:** what the run used. **Out:** peak GPU memory (per GPU), CPU memory and tokens per second. **Why this way:** a few lines now save a wrong plan later.

Note: this is **writing** speed for one answer at a time. Training speed and batched writing are different numbers.

In [ ]:
print("load mode:", LOAD_MODE)
print("peak GPU memory (GB):", [round(torch.cuda.max_memory_reserved(i) / 1024**3, 3) for i in range(torch.cuda.device_count())])
print("CPU memory used by this notebook (GB):", round(psutil.Process().memory_info().rss / 1024**3, 3))
if rows:
    print("writing speed (tokens/second):", round(sum(r["tokens_per_second"] for r in rows) / len(rows), 1))
print("\nTell these numbers to your supervisor / write them into results/ and DECISIONS.md")